# Notebook 05 – Integração com Oracle Cloud Object Storage

Este notebook tem como objetivo integrar os artefatos do projeto (datasets e modelos) armazenados no Google Drive ao Oracle Cloud Infrastructure (OCI) Object Storage. Ele centraliza a autenticação, upload, download e validação dos arquivos, servindo como pipeline de publicação entre a etapa de Ciência de Dados e o backend da aplicação.

Instalando o SDK no Colab

In [1]:
!pip install oci

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.0/36.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


Montagem do Drive

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Variáveis para não repetir o caminho

In [3]:
BASE_PATH = "/content/drive/MyDrive/Hackathon_OCI_G9/HACKATHON G9 - TEAM 20"

DATASETS_PATH = f"{BASE_PATH}/datasets"
MODELOS_PATH = f"{BASE_PATH}/modelos"

Verificar se as pastas existem

In [4]:
import os

print(os.path.exists(DATASETS_PATH))
print(os.path.exists(MODELOS_PATH))

True
True


Verificar o conteúdo

In [5]:
for arquivo in os.listdir(DATASETS_PATH):
    print(arquivo)

sidra_pof_gastos_alimentares.csv
perfis_financeiros_sinteticos.csv
transacoes_sinteticas.csv
bcb_inadimplencia_pf.csv
sidra_despesas_limpo.csv
sidra_despesas_bruto.csv
sidra_despesas_por_categoria_renda.csv


In [6]:
for arquivo in os.listdir(MODELOS_PATH):
    print(arquivo)

modelo_perfil_producao.pkl
escalador_perfil.pkl
codificador_perfil.pkl
modelo_perfil_deep_learning.h5
tokenizador_categoria.pkl
modelo_categoria_deep_learning.h5
modelo_categoria_producao.pkl
codificador_categorias.pkl
vetorizador_tfidf.pkl


Criando o arquivo config

In [7]:
config_content = """
[DEFAULT]

# Informe o OCID do usuário da OCI
user=<USER_OCID>

# Informe o fingerprint da API Key cadastrada
fingerprint=<FINGERPRINT>

# Informe o OCID da Tenancy
tenancy=<TENANCY_OCID>

# Região da OCI (ex.: sa-saopaulo-1)
region=<REGION>

# Caminho para a chave privada (.pem)
key_file=<CAMINHO_DA_CHAVE_PRIVADA>
"""

with open("/content/config", "w") as f:
    f.write(config_content)

print("Arquivo config criado com sucesso!")

Arquivo config criado com sucesso!


Testando a autenticação

In [8]:
import oci

config = oci.config.from_file("/content/config")

object_storage = oci.object_storage.ObjectStorageClient(config)

namespace = object_storage.get_namespace().data

print("Namespace:", namespace)


Namespace: grrjqrgpt6r9


Listando os buckets

In [13]:
config = oci.config.from_file("/content/config")

object_storage = oci.object_storage.ObjectStorageClient(config)

namespace = object_storage.get_namespace().data
compartment_id = "ocid1.compartment.oc1..aaaaaaaauo43xyzqyjuu4ju3mnasqs6sird4x37xpiqlomw2nvyy2d4f4k3a"

buckets = object_storage.list_buckets(
    namespace_name=namespace,
    compartment_id=compartment_id
)

for bucket in buckets.data:
    bucket_name = bucket.name
    print(bucket_name)

hackathon-one-g9-team-20


Função para fazer upload arquivo por arquivo

In [16]:
def upload_file(local_path, object_name):

    with open(local_path, "rb") as arquivo:

        object_storage.put_object(
            namespace_name=namespace,
            bucket_name=bucket_name,
            object_name=object_name,
            put_object_body=arquivo
        )

    print(f"✔ {object_name}")

Função para fazer upload do diretório

In [14]:
import os

def upload_directory(local_directory, bucket_directory):
    """
    Envia todos os arquivos de uma pasta para um diretório do bucket.

    Exemplo:

    upload_directory(
        "/content/datasets",
        "datasets"
    )
    """

    arquivos = os.listdir(local_directory)

    print(f"\nEncontrados {len(arquivos)} arquivos.\n")

    for arquivo in arquivos:

        caminho_local = os.path.join(
            local_directory,
            arquivo
        )

        if not os.path.isfile(caminho_local):
            continue

        object_name = f"{bucket_directory}/{arquivo}"

        upload_file(
            caminho_local,
            object_name
        )

    print("\nUpload finalizado.")

Enviando todos os arquivos

In [ ]:
for arquivo in os.listdir(DATASETS_PATH):

    caminho = os.path.join(DATASETS_PATH, arquivo)

    if os.path.isfile(caminho):

        upload_file(
            caminho,
            f"datasets/{arquivo}"
        )

Enviando os diretórios

In [17]:
upload_directory(DATASETS_PATH, "datasets")

upload_directory(MODELOS_PATH, "models")


Encontrados 7 arquivos.

✔ datasets/sidra_pof_gastos_alimentares.csv
✔ datasets/perfis_financeiros_sinteticos.csv
✔ datasets/transacoes_sinteticas.csv
✔ datasets/bcb_inadimplencia_pf.csv
✔ datasets/sidra_despesas_limpo.csv
✔ datasets/sidra_despesas_bruto.csv
✔ datasets/sidra_despesas_por_categoria_renda.csv

Upload finalizado.

Encontrados 9 arquivos.

✔ models/modelo_perfil_producao.pkl
✔ models/escalador_perfil.pkl
✔ models/codificador_perfil.pkl
✔ models/modelo_perfil_deep_learning.h5
✔ models/tokenizador_categoria.pkl
✔ models/modelo_categoria_deep_learning.h5
✔ models/modelo_categoria_producao.pkl
✔ models/codificador_categorias.pkl
✔ models/vetorizador_tfidf.pkl

Upload finalizado.


Conferindo o Bucket

In [18]:
objects = object_storage.list_objects(
    namespace_name=namespace,
    bucket_name=bucket_name
)

for obj in objects.data.objects:
    print(obj.name)

datasets/bcb_inadimplencia_pf.csv
datasets/perfis_financeiros_sinteticos.csv
datasets/sidra_despesas_bruto.csv
datasets/sidra_despesas_limpo.csv
datasets/sidra_despesas_por_categoria_renda.csv
datasets/sidra_pof_gastos_alimentares.csv
datasets/transacoes_sinteticas.csv
models/codificador_categorias.pkl
models/codificador_perfil.pkl
models/escalador_perfil.pkl
models/modelo_categoria_deep_learning.h5
models/modelo_categoria_producao.pkl
models/modelo_perfil_deep_learning.h5
models/modelo_perfil_producao.pkl
models/tokenizador_categoria.pkl
models/vetorizador_tfidf.pkl
